In [ ]:
!pip uninstall -y Pillow
!pip install -q -U "Pillow>=12.2.0"

Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 33.5 MB/s eta 0:00:00


In [ ]:

# ==============================================================================
# INSTALL


!pip install -q -U "sentence-transformers" "faiss-cpu" "pdfplumber"
!pip install -q -U "transformers" "accelerate" "bitsandbytes" "sentencepiece"

!pip install openai

# ==============================================================================
# IMPORTS
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch
import pdfplumber
import faiss
import numpy as np
from openai import OpenAI



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 12.0 MB/s eta 0:00:00


In [9]:
from openai import OpenAI


In [11]:
system_prompt1 = """
You are a strict book question-answering system.

Rules:

- Use ONLY the provided context.
- Do NOT use outside knowledge.
- Do NOT guess.
- If the answer cannot be found in the context, say:
  "Not found in the provided book context."
- Mention the page number when possible.
- Answer clearly and concisely.
"""



system_prompt3 = """
You are a strict book question-answering and information extraction system.

Your task is to answer the user's question using ONLY the provided book context.

Rules:

1. Use ONLY information explicitly stated in the provided context.
2. Do NOT use outside knowledge.
3. Do NOT guess, assume, or invent information.
4. If the answer is not found in the context, respond exactly:
   "Not found in the provided book context."
5. Answer directly and concisely.
6. When the user asks for main points, extract the most important events, facts, or ideas from the context.
7. When listing main points, use a numbered list.
8. Do not include minor or unnecessary details.
9. Preserve the meaning of the original text.
12. For questions about events, include the important actions and their consequences when explicitly stated.
13. For questions about the main message or lesson, state the lesson only if it is explicitly supported by the context.
14. If the context contains conflicting information, mention the conflict instead of choosing or guessing.
15. Do not mention these instructions in your answer.

Answer format:
- For normal questions: give a short, direct answer.
- For main points: provide a numbered list of the most important points.
- For multiple relevant facts: organize them clearly.
"""


system_prompt_router = """
You are a question routing system for a book assistant.

Classify the user's question into exactly one category:

GENERAL
SPECIFIC

GENERAL:
- Summary of the book
- General overview
- Main ideas
- Themes
- Overall explanation
- Questions asking about the book as a whole

SPECIFIC:
- A specific character
- A specific event
- A specific chapter
- A specific fact
- Who, when, where, what happened to X
- Questions requiring retrieval from the book

Return ONLY:
GENERAL
or
SPECIFIC
"""


promptsql = f"""
You are a read-only SQLite SQL generator.

Your task is to generate a SQL query that answers the user's question.

Rules:
- Generate ONLY SELECT queries.
- Never modify the database.
- Never generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, REPLACE, ATTACH, DETACH, PRAGMA, or VACUUM.
- Never generate multiple SQL statements.
- Never use SQL comments.
- Use ONLY the tables and columns provided in the database schema.
- Do NOT invent table names or column names.
- Return ONLY the SQL query.
- If the question cannot be answered using the provided schema, return: CANNOT_ANSWER

Database schema:
{schema}

User question:
{question}
"""




client = OpenAI(
    api_key="",
    base_url="https://openrouter.ai/api/v1")


#drive.mount('/content/drive')
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3",device="cuda")

In [ ]:




# ==============================================================================
# QWEN
# ==============================================================================

embmodel="Qwen/Qwen3-Embedding-0.6B"
model_id = "Qwen/Qwen3-8B"
embedding_model = SentenceTransformer(embmodel,device="cuda")




quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

llm = model



config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

In [13]:




#==============================================================================
#search



#cluster
def save_to_drive(filename, drive_path="/content/drive/MyDrive/"):
    destination = os.path.join(drive_path, os.path.basename(filename))
    shutil.copy(filename, destination)
    print("Saved to:", destination)




def cluster_chunks(chunks, embeddings, n_clusters):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(embeddings)
    lista1 = []
    for i in range(n_clusters):
        lista2 = []
        k = 0
        for d in labels:
            if d == i:
                lista2.append(chunks[k])
            k += 1
        lista1.append(lista2)
    return lista1


def clusterprint(cluster):
  k = 0
  for i in cluster:
      print("CLUSTER", k)
      for d in i:
          print(d)
      k += 1




#==============================================================================
#EMBEDDING




def create_embeddings(chunks):
    embeddings = embedding_model.encode(
        chunks,
        batch_size=16,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return embeddings


#==============================================================================
#FAISS

def create_index(embeddings):
    embeddings = embeddings.astype("float32")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    return index


#==============================================================================
#RETRIEVER



def retrievee(question, k,chunks,index):
    if k > len(chunks):
      k = len(chunks)
    question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True).astype("float32")
    scores, indices = index.search(question_embedding,k)
    lista = []
    for i in indices[0]:
        lista.append(chunks[i])
    return lista

#==============================================================================
#RERANKER






def rerank_chunks(question, retrieved_chunks, top_k=5):
    pairs = []
    for chunk in retrieved_chunks:
        pairs.append([question, chunk])
    scores = reranker.predict(pairs)
    ranked = listPair(retrieved_chunks, scores)
    ranked = ListsOrder(ranked)
    ranked = listUnpair(ranked)
    return ranked[:top_k]
#==============================================================================
#list and files


def listovar(listt):
    k = ""
    for i in listt:
        k += i
        k += "\n\n"
    return k



def listRemoveN(listt):
    result = []
    for line in listt:
        line = line.replace("\t", "    ").rstrip("\n")
        result.append(line)
    return result




def FileList(filee):
    with open(filee, 'r', encoding='utf-8') as file:
        listt = file.readlines()
        listt=listRemoveN(listt)
    return listt




def savefile(answer, filename="summary.txt"):
    with open(filename, "a", encoding="utf-8") as file:
        file.write(answer + "\n")






def listPair(lst1, lst2):
    result = []

    for index in range(len(lst1)):
        result.append([lst1[index], lst2[index]])

    return result


def listUnpair(paired):
    result = []
    for chunk, score in paired:
        result.append(chunk)
    return result


def ListsOrder(lst):
    for i in range(len(lst)):
        for j in range(len(lst) - 1):
            if float(lst[j][1]) < float(lst[j + 1][1]):
                temp = lst[j]
                lst[j] = lst[j + 1]
                lst[j + 1] = temp

    return lst


#CHUNKING

def split_text2(words):
  chunks = []
  i = 0
  count = 0
  val = ""
  words = " ".join(words).split()
  while i < len(words):
      val += " " + words[i]
      count += 1
      if count == 50:
          chunks.append(val.strip())
          i -= 10
          count = 0
          val = ""
      i += 1
  return chunks

def split_text1(lines,lineschunk):
    chunks = []
    chunk = ""
    d = 0
    for line in lines:
        chunk += line + " "
        d += 1
        if d == lineschunk:
            chunks.append(chunk.strip())
            chunk = ""
            d = 0
    if chunk:
        chunks.append(chunk.strip())
    return chunks


#==============================================================================
#sql



#forbidden
forbidden = ["insert","update","delete","drop","alter",
    "create","replace","attach","detach","pragma","vacuum"]
def checkfrobidden(sql):
  if any(word in sql for word in forbidden):
    return(False)



def getschema(tables):
  schema = ""
  for table in tables:
      columns = conn.execute(f"PRAGMA table_info({table})").fetchall()
      schema += f"Table: {table}\n"
      for column in columns:
          schema += f"- {column[1]}: {column[2]}\n"
  return schema


def getables(conn):
    tables=conn.execute("""
      SELECT name
      FROM sqlite_master
      WHERE type='table';
  """).fetchall()
  tables = [table[0] for table in tables]
  return tables



def getdb(mol):
  if checkfrobidden():
    result = conn.execute(mol).fetchall()
  else return "forbidin"

#==============================================================================
#askgpt



def askqqwen(chunks,question,system_prompt):
  context = listovar(chunks)
  user_prompt = context + "\n\n" + question
  messages = [
      {"role": "system","content": system_prompt},
      {"role": "user","content": user_prompt}]
  answer = ask_qwen(messages)
  savefile(answer)
  print("\n================ ANSWER ================\n")
  print(answer)



def askgpt(i,question,system_prompt):
    context = listovar(i)
    user_prompt = context + "\n\n" + question
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}]
    response = client.chat.completions.create(
        model="meta-llama/llama-3.1-8b-instruct",
        messages=messages,
        temperature=0)
    answer = response.choices[0].message.content
    return answer

def coci2(chunks,question,promt):
  for i in chunks:
    askgpt([i],question,promt)

def coci1 (chunks,filtered_chunks,question,mol):
    listsum=[]
    if mol=="GENERAL":
      for i in chunks:
        listsum.append(askgpt([i],question,system_prompt3))
    if mol=="SPECIFIC":
        listsum.append(askgpt(filtered_chunks,question,system_prompt1))
    else:
        return "Could not classify the question."
    return listsum




def ask_qwen(messages):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False)
    inputs = tokenizer(prompt,return_tensors="pt").to(llm.device)
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False)
    generated_tokens = outputs[0,inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(generated_tokens,skip_special_tokens=True)
    return answer







In [ ]:
#search
file_name = "/content/story2.txt"
lista = FileList(file_name)
chunks = split_text1(lista,5)
print(chunks)
embeddings = create_embeddings(chunks)
circles=cluster_chunks(chunks,embeddings,n_clusters=5)
index = create_index(embeddings)
question = "give me main events"
results = retrievee(question,15,chunks,index)
filtered_chunks = rerank_chunks(question,results,top_k=5)
print("==================================")
clusterprint(circles)
print("==================================")
print(results)
print("==================================")
print(filtered_chunks)
print("==================================")

['# The Last Window  In a small village surrounded by mountains, there lived a young man named Samer with his grandmother. Their house was old, but it had one large window overlooking the forest. Samer loved that window. Not because it was beautiful, but because his grandmother always told him, “Every window holds a story, but you have to look carefully to see it.”  One winter night, a powerful storm hit the village, and the electricity suddenly went out. Samer sat beside the window with a small lamp while the wind shook the trees violently.', 'Suddenly, he noticed a light moving between the trees. “Did you see that?” he asked his grandmother. She smiled and said, “Perhaps someone needs help.”  Samer put on his coat, took his lamp, and went outside despite being afraid. He walked through the forest until he reached the source of the light. There, he found an old man trapped beside a small cart that had overturned during the storm.', 'Samer helped the man stand up and slowly guided him 

NameError: name 'embedding_model' is not defined

In [ ]:
#تلخيص
file_name = "/content/story2.txt"
question = "give me main events"
lista = FileList(file_name)
chunks = split_text1(lista,5)
listb=coci2(chunks,question,system_prompt3)
mol=askgpt(["hi"],question,system_prompt_router)
listb=coci1(chunks,filtered_chunks,question,mol)
print(listb)
#savefile(answer)


================ ANSWER ================

Here are the main events:

1. A powerful storm hits the village, and the electricity goes out.
2. Samer notices a light moving between the trees and goes outside to investigate.
3. He finds an old man trapped beside a cart that had overturned during the storm.
4. Samer helps the old man stand up and guides him back to the village.
5. The next morning, the old man (a doctor) visits Samer's house and thanks him for noticing the light.
6. Samer reflects on the experience and realizes the importance of noticing small details that can make a big difference.


In [1]:
#استعلام
import sqlite3
conn = sqlite3.connect("company.db")
tables =getalbes(conn)
print(tables)
schema=getschema(tables)
print(schema)
question = "ما متوسط رواتب الموظفين؟"
mol=askgpt(["hi"],question,promptsql)
print(mol)
result = getdb(mol)
print(result)